# Imports and Configuration

In [1]:
import re, gc, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from surprise import SVD, Dataset, Reader

warnings.filterwarnings("ignore")

config

In [2]:
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)

MIN_BOOK_RATINGS = 100
MIN_USER_RATINGS = 20
N_USERS = 100000                # number of sample (1M is too large -> not enough RAM -> explosion)
RELEVANT_RATING = 4             # threshold for "liked"
K = 10                          # top-K for Precision@K / Recall@K
EVAL_USERS = 1000
LIKED_LIMIT = 20
N_NEIGHBORS = 50
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

### Load Data

In [3]:
interactions = pd.read_parquet(r"C:\Users\Quang\Documents\USTH\FundDS\dataset\50ubr.parquet")

book_cols = ["book_id", "title", "author_names"] + ["genre_1", "genre_2", "genre_3", "genre_4", "genre_5"] + ["average_rating", "ratings_count"]
books = pd.read_parquet(r"C:\Users\Quang\Documents\USTH\FundDS\dataset\mart_book_statistics.parquet", columns=book_cols).drop_duplicates("book_id")

print(f"Interactions: {interactions.shape}")
print(f"Books: {books.shape}")

Interactions: (50000000, 3)
Books: (2363585, 10)


### Data Cleaning

In [4]:
df = interactions[interactions["book_id"].isin(books["book_id"])]
df = df.drop_duplicates(["user_id", "book_id"])
del interactions
gc.collect()

0

In [5]:
book_counts = df["book_id"].value_counts()
df = df[df["book_id"].isin(book_counts[book_counts >= MIN_BOOK_RATINGS].index)]

user_counts = df["user_id"].value_counts()
users_clean = user_counts.index.to_numpy()
if len(users_clean) > N_USERS:
    users_clean = rng.choice(users_clean, N_USERS, replace=False)
df = df[df["user_id"].isin(users_clean)].copy()

In [6]:
# convert author to list of strings
def to_list(x):                 
    if x is None:
        return []
    if isinstance(x, str):
        return [x]
    try:
        return [str(a) for a in x if a is not None]
    except TypeError:         
        return []

# matrix
df["user_idx"], user_ids = pd.factorize(df["user_id"], sort=True)
df["item_idx"], book_ids = pd.factorize(df["book_id"], sort=True)
df["rating"] = df["rating"].astype("float32")
n_users, n_items = len(user_ids), len(book_ids)

books_meta = books.set_index("book_id").loc[book_ids].copy()
books_meta["author"] = books_meta["author_names"].apply(lambda x: ", ".join(to_list(x)))
del books
gc.collect()

print(f"Users: {n_users:,}")
print(f"Books: {n_items:,}")
print(f"Ratings: {len(df):,}\n")
print(f"Matrix Density: {len(df) / (n_users * n_items):.4%}\n")
print(df["rating"].value_counts().sort_index().to_string())

Users: 100,000
Books: 46,151
Ratings: 8,315,448

Matrix Density: 0.1802%

rating
1.0     169519
2.0     468083
3.0    1745703
4.0    2957808
5.0    2974335


# Model

### Train / validation / test split

Split per user

Example: (User has 20 rating -> 16 train / 2 val / 2 test)

In [7]:
df = df.sample(frac=1, random_state=RANDOM_STATE)
position = df.groupby("user_idx").cumcount()  
size = df.groupby("user_idx")["user_idx"].transform("size")  
frac     = position / size

train = df[frac < 0.8]
val = df[(frac >= 0.8) & (frac < 0.9)]
test = df[frac >= 0.9]

print(f"Train: {len(train):,}")
print(f"Valid: {len(val):,}")
print(f"Test: {len(test):,}")

Train: 6,692,331
Valid: 836,479
Test: 786,638


In [8]:
# user-item matrix from train
R_train   = csr_matrix((train["rating"], (train["user_idx"], train["item_idx"])), shape=(n_users, n_items))
item_user = R_train.T.tocsr()

### Evaluation setup

In [9]:
def group_items(data):
    return data.groupby("user_idx")["item_idx"].agg(list).to_dict()

# group book that user like (rating > 4)
liked_train = group_items(train[train["rating"] >= RELEVANT_RATING])
liked_val = group_items(val[val["rating"] >= RELEVANT_RATING])
liked_test = group_items(test[test["rating"] >= RELEVANT_RATING])

# eligible user = user that in both liked train/val/test
eligible   = [u for u in liked_train if u in liked_val and u in liked_test]
eval_users = rng.choice(eligible, min(EVAL_USERS, len(eligible)), replace=False)

# books that user has readed
sub = lambda d: d[d["user_idx"].isin(eval_users)]
readed_train = group_items(sub(train))
readed_val = group_items(sub(val))
readed_for_val = {u: readed_train[u] for u in eval_users}
readed_for_test = {u: readed_train[u] + readed_val[u] for u in eval_users}

In [10]:
def user_metrics(scores, u):
    out = []
    for seen, relevant in ((readed_for_val[u], liked_val[u]), (readed_for_test[u], liked_test[u])):
        s = scores.astype("float64").copy()
        s[seen] = -np.inf                              # already-seen books -> minus score
        top = np.argpartition(-s, K)[:K]
        hits = np.isin(top, relevant).sum()
        out += [hits / K, hits / len(relevant)]
    return np.array(out)

def evaluate(score_fn):
    return np.mean([user_metrics(score_fn(u), u) for u in eval_users], axis=0)

def rating_errors(pred, true):
    err = np.asarray(pred) - np.asarray(true)
    return np.abs(err).mean(), np.sqrt((err ** 2).mean())

results = {}
def save(name, m, mae=None, rmse=None):
    results[name] = dict(val_P=m[0], val_R=m[1], test_P=m[2], test_R=m[3], MAE=mae, RMSE=rmse)

## **Popularity Baseline**

In [11]:
stats = train.groupby("item_idx")["rating"].agg(["count", "mean"]).reindex(range(n_items))
v = stats["count"].fillna(0).to_numpy()             # number of rating of the book
R = stats["mean"].fillna(0).to_numpy()              # avg rating of the book
C = train["rating"].mean()                          # avg rating of all the books
m = np.percentile(v[v > 0], 80)                     # minimum votes threshold

pop_score = v / (v + m) * R + m / (v + m) * C 

mae, rmse = rating_errors(pop_score[test["item_idx"]], test["rating"])

def pop_fn(user):
    return pop_score

save("Popularity", evaluate(pop_fn), mae, rmse)
pd.Series(results["Popularity"])

val_P     0.007300
val_R     0.013904
test_P    0.006800
test_R    0.013215
MAE       0.751069
RMSE      0.950471
dtype: float64

## **Item-Based Collaborative Filtering**

Idea:  user: like books -> recommendations most similar books (cosine similarity)

In [12]:
def make_neighbor_fn(X, nn):

    def score_fn(u):
        seeds = liked_train.get(u, [])[:LIKED_LIMIT]        # take limit 20 books that user like
        scores = np.zeros(X.shape[0])
        if len(seeds) == 0:
            return scores

        # find n_neighbors: Cosine Distance -> Cosine Similarity
        dist, idx = nn.kneighbors(X[seeds], n_neighbors=N_NEIGHBORS)    
        scores += np.bincount(idx.ravel(), weights=(1 - dist).ravel(), minlength=X.shape[0])
        return scores
    
    return score_fn

nn_cf = NearestNeighbors(metric="cosine", algorithm="brute").fit(item_user)   # works on sparse data
itemcf_fn = make_neighbor_fn(item_user, nn_cf)

save("Item-CF", evaluate(itemcf_fn))
pd.Series(results["Item-CF"])

val_P     0.083200
val_R     0.155123
test_P    0.084000
test_R    0.166447
MAE            NaN
RMSE           NaN
dtype: float64

## **SVD Matrix Factorization**

In [13]:
reader   = Reader(rating_scale=(1, 5))
trainset = Dataset.load_from_df(train[["user_idx", "item_idx", "rating"]], reader).build_full_trainset()
val_list  = list(zip(val["user_idx"].tolist(),  val["item_idx"].tolist(),  val["rating"].tolist()))
test_list = list(zip(test["user_idx"].tolist(), test["item_idx"].tolist(), test["rating"].tolist()))

def svd_errors(algo, data):
    est = np.array([p.est for p in algo.test(data)])
    return rating_errors(est, [r for _, _, r in data])

svd_models, rows = {}, []
for f in [20, 50, 100]:                                    # only 3 simple configurations
    algo = SVD(n_factors=f, n_epochs=15, random_state=RANDOM_STATE)
    algo.fit(trainset)                                     # trained on TRAIN only
    mae_v, rmse_v = svd_errors(algo, val_list)             # validation error
    svd_models[f] = algo
    rows.append({"n_factors": f, "val_MAE": mae_v, "val_RMSE": rmse_v})

svd_table = pd.DataFrame(rows)
best_f = int(svd_table.loc[svd_table["val_RMSE"].idxmin(), "n_factors"])
print(svd_table.round(4).to_string(index=False))
print("selected n_factors (lowest validation RMSE):", best_f)

 n_factors  val_MAE  val_RMSE
        20   0.6590    0.8461
        50   0.6566    0.8439
       100   0.6557    0.8438
selected n_factors (lowest validation RMSE): 100


In [14]:
svd = svd_models[best_f]
mae, rmse = svd_errors(svd, test_list)

raw_items = [int(trainset.to_raw_iid(i)) for i in range(trainset.n_items)]
def svd_fn(u):
    s = svd.bi + svd.qi @ svd.pu[trainset.to_inner_uid(int(u))]
    full = np.full(n_items, s.min())
    full[raw_items] = s
    return full

save("SVD", evaluate(svd_fn), mae, rmse)
pd.Series(results["SVD"])

val_P     0.000700
val_R     0.001134
test_P    0.000500
test_R    0.000614
MAE       0.654782
RMSE      0.843297
dtype: float64

## **Content-Based Filtering**

In [15]:
def clean(token):
    """'Haruki Murakami' -> 'haruki_murakami' (one author / genre = one token)"""
    return re.sub(r"\W+", "_", str(token).lower()).strip("_")

genre_text = books_meta[["genre_1", "genre_2", "genre_3", "genre_4", "genre_5"]].apply(
    lambda row: " ".join(clean(g) for g in row if pd.notna(g) and clean(g)), axis=1)

author_text = books_meta["author_names"].apply(
    lambda x: " ".join(t for t in (clean(a) for a in to_list(x)) if t))

def build_content_fn(texts):
    X  = TfidfVectorizer(token_pattern=r"\S+", lowercase=False).fit_transform(texts)
    nn = NearestNeighbors(metric="cosine", algorithm="brute").fit(X)
    return make_neighbor_fn(X, nn), X.shape[1]

### Genre

In [16]:
genre_fn, size = build_content_fn(genre_text)
save("CB (Genre)", evaluate(genre_fn))
print("vocabulary size:", size); pd.Series(results["CB (Genre)"])

vocabulary size: 16


val_P     0.003200
val_R     0.006250
test_P    0.003300
test_R    0.005706
MAE            NaN
RMSE           NaN
dtype: float64

### Author

In [17]:
author_fn, size = build_content_fn(author_text)
save("CB (Author)", evaluate(author_fn))
print("vocabulary size:", size); pd.Series(results["CB (Author)"])

vocabulary size: 18784


val_P     0.012000
val_R     0.024151
test_P    0.010600
test_R    0.022690
MAE            NaN
RMSE           NaN
dtype: float64

### Genre + Author

In [18]:
both_fn, size = build_content_fn(genre_text + " " + author_text)
save("CB (Genre + Author)", evaluate(both_fn))
print("vocabulary size:", size); pd.Series(results["CB (Genre + Author)"])

vocabulary size: 18800


val_P     0.011500
val_R     0.023464
test_P    0.011900
test_R    0.029056
MAE            NaN
RMSE           NaN
dtype: float64

# Model Comparison

In [19]:
def comparison_table(split="test"):
    rows = []
    for name, r in results.items():
        f = lambda x: "-" if x is None else round(float(x), 4)
        rows.append({"Model": name, "MAE": f(r["MAE"]), "RMSE": f(r["RMSE"]),
                     "Precision@10": f(r[f"{split}_P"]), "Recall@10": f(r[f"{split}_R"])})
    return pd.DataFrame(rows).set_index("Model")

In [20]:
comparison_table("val")

,MAE,RMSE,Precision@10,Recall@10
Model,,,,
Popularity,0.7511,0.9505,0.0073,0.0139
Item-CF,-,-,0.0832,0.1551
SVD,0.6548,0.8433,0.0007,0.0011
CB (Genre),-,-,0.0032,0.0063
CB (Author),-,-,0.0120,0.0242
CB (Genre + Author),-,-,0.0115,0.0235


In [21]:
comparison_table("test")

,MAE,RMSE,Precision@10,Recall@10
Model,,,,
Popularity,0.7511,0.9505,0.0068,0.0132
Item-CF,-,-,0.0840,0.1664
SVD,0.6548,0.8433,0.0005,0.0006
CB (Genre),-,-,0.0033,0.0057
CB (Author),-,-,0.0106,0.0227
CB (Genre + Author),-,-,0.0119,0.0291


## **Hybrid Model**

In [26]:
# Scale score to 0-1 per user
def normalize(s):
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else np.zeros_like(s, dtype="float64")

alphas = [0.25, 0.50, 0.75, 0.9]
sums = {a: np.zeros(4) for a in alphas}
for u in eval_users:
    cf = normalize(itemcf_fn(u))
    ct = normalize(both_fn(u))
    for a in alphas:
        sums[a] += user_metrics(a * cf + (1 - a) * ct, u) 

alpha_table = pd.DataFrame({a: sums[a] / len(eval_users) for a in alphas}, index=["val_P@10", "val_R@10", "test_P@10", "test_R@10"]).T
alpha_table.index.name = "alpha"
best_alpha = alpha_table["val_P@10"].idxmax()  

print(alpha_table.round(4).to_string())
print("Selected alpha:", best_alpha)

save(f"Hybrid, alpha={best_alpha})", alpha_table.loc[best_alpha].to_numpy())
comparison_table("test")

       val_P@10  val_R@10  test_P@10  test_R@10
alpha                                          
0.25     0.0364    0.0721     0.0324     0.0777
0.50     0.0589    0.1175     0.0559     0.1240
0.75     0.0906    0.1743     0.0899     0.1830
0.90     0.0876    0.1661     0.0872     0.1750
Selected alpha: 0.75


,MAE,RMSE,Precision@10,Recall@10
Model,,,,
Popularity,0.7511,0.9505,0.0068,0.0132
Item-CF,-,-,0.0840,0.1664
SVD,0.6548,0.8433,0.0005,0.0006
CB (Genre),-,-,0.0033,0.0057
CB (Author),-,-,0.0106,0.0227
CB (Genre + Author),-,-,0.0119,0.0291
"Hybrid (<function make_neighbor_fn.<locals>.score_fn at 0x0000026B3534E560> + <function make_neighbor_fn.<locals>.score_fn at 0x0000026DCC5000F0>, alpha=0.75)",-,-,0.0899,0.1830
"Hybrid, alpha=0.75)",-,-,0.0899,0.1830


## 12. Final Recommendation Demo

In [ ]:
user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
all_rated = df.groupby("user_idx")["item_idx"].agg(list).to_dict()

def hybrid_scores(u):
    return best_alpha * normalize(itemcf_fn(u)) + (1 - best_alpha) * normalize(both_fn(u))

def recommend_books(user_id, n=10):
    # if user not in data -> popularity list
    if user_id not in user_to_idx:
        scores, seen = pop_score.copy(), []
    else:
        u = user_to_idx[user_id]
        scores, seen = hybrid_scores(u), all_rated[u]
    scores = scores.astype("float64")
    scores[seen] = -np.inf
    
    top = np.argsort(-scores)[:n]
    info = books_meta.iloc[top]

    recommend10 = pd.DataFrame({
        "Rank": range(1, n + 1), "book_id": book_ids[top], "Title": info["title"].values,
        "Author": info["author"].values, "Score": scores[top].round(3),
        "Avg Rating": info["average_rating"].values, "Ratings Count": info["ratings_count"].values})
    return recommend10


In [31]:
user = user_ids[eval_users[0]]

print(f"\nTop-10 recommendations for user {user}:")
recommend_books(user, n=10)


Top-10 recommendations for user c6226744205f7b80c5af3d207761d0f2:


,Rank,book_id,Title,Author,Score,Avg Rating,Ratings Count
0,1,7613,Animal Farm,George Orwell,0.622,4.03,4809015.0
1,2,11,The Hitchhiker's Guide to the Galaxy (Hitchhik...,Douglas Adams,0.550,4.20,940154.0
2,3,2956,The Adventures of Huckleberry Finn,"Guy Cardwell, John Seelye, Mark Twain",0.519,3.80,969291.0
3,4,2657,To Kill a Mockingbird,Harper Lee,0.519,4.26,7123019.0
4,5,890,Of Mice and Men,John Steinbeck,0.486,3.90,2951715.0
5,6,3,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPre",0.477,4.47,11810403.0
6,7,5129,Brave New World,Aldous Huxley,0.462,3.97,1043619.0
7,8,4671,The Great Gatsby,F. Scott Fitzgerald,0.454,3.89,2758812.0
8,9,4381,Fahrenheit 451,Ray Bradbury,0.436,3.97,591506.0
9,10,4981,Slaughterhouse-Five,Kurt Vonnegut Jr.,0.426,4.06,861654.0
